# Saudi Plate OCR — Train on Kaggle

Same pipeline as the Colab version, adapted for Kaggle (~30 GPU hrs/week).

## ONE-TIME SETUP (do this before running cells)
In the right-hand panel:
1. **Session options → Accelerator → `GPU T4 x2`** (or P100).
2. **Session options → Internet → On** (needed for pip/clone/download).
3. **Add-ons → Secrets → Add a new secret**: name `GH_TOKEN`, value = your GitHub token (repo read). Toggle it **Attached** to this notebook.

Then run cells top to bottom.


## 1. Install
Paddle 2.6.2 (CUDA 11.8) + PaddleOCR 2.7 + NumPy<2. ~3-4 min.


In [ ]:
!apt-get -qq update && apt-get -qq -y install fonts-noto-core fonts-kacst fonts-hosny-amiri fonts-dejavu-core > /dev/null 2>&1
!pip install -q paddlepaddle-gpu==2.6.2 --extra-index-url https://www.paddlepaddle.org.cn/packages/stable/cu118/
!rm -rf PaddleOCR
!git clone -q --depth 1 -b release/2.7 https://github.com/PaddlePaddle/PaddleOCR.git
!pip install -q -r PaddleOCR/requirements.txt
!pip install -q 'paddle2onnx==1.1.0' onnxruntime
!pip install -q 'numpy<2'
import paddle
print('paddle', paddle.__version__, '| GPU:', paddle.device.is_compiled_with_cuda())
assert paddle.device.is_compiled_with_cuda(), 'No GPU — set Accelerator to GPU in Session options'


## 2. Get our code from GitHub
Reads the `GH_TOKEN` secret (Kaggle's secret system, not Colab's).


In [ ]:
from kaggle_secrets import UserSecretsClient
import subprocess
TOKEN = UserSecretsClient().get_secret('GH_TOKEN').strip()
subprocess.run('rm -rf saudi-plate-ocr', shell=True)
subprocess.run(['git','clone','-q',f'https://{TOKEN}@github.com/Tanzimalam1454/saudi-plate-ocr.git'])
for f in ['plate_spec.py','generate_plates.py','saudi_rec_config.yml','saudi_rec_infer.py','prepare_real_data.py','build_real_labels.py','prepare_char_dataset.py']:
    subprocess.run(['cp', f'saudi-plate-ocr/{f}', '.'])
import os; assert all(os.path.exists(f) for f in ['plate_spec.py','generate_plates.py','saudi_rec_config.yml'])
print('files in place')


## 3. Generate synthetic data (multi-core, ~5 min)
8,000 plates -> 16,000 labeled half-crops, train/val split.


In [ ]:
!rm -rf dataset
!python generate_plates.py --count 8000 --out dataset --montage
!wc -l dataset/train_label.txt dataset/val_label.txt
from IPython.display import Image as IPImage; IPImage('dataset/_montage.jpg', width=900)


## 4. Download pretrained Arabic recognizer


In [ ]:
import os, glob
if not glob.glob('pretrain/arabic_PP-OCRv*_rec_train'):
    os.makedirs('pretrain', exist_ok=True)
    !cd pretrain && wget -q https://paddleocr.bj.bcebos.com/PP-OCRv4/multilingual/arabic_PP-OCRv4_rec_train.tar && tar xf arabic_PP-OCRv4_rec_train.tar
!ls pretrain


## 5. Train (saves best model each epoch)
Watch the `best metric, acc:` line. Stop when it plateaus — best model is kept in
`output/saudi_rec/best_accuracy`.


In [ ]:
import glob, re
pre = glob.glob('pretrain/arabic_PP-OCRv*_rec_train')[0] + '/best_accuracy'
n_train = sum(1 for l in open('dataset/train_label.txt') if l.strip())
iters = max(1, n_train // 64)
cfg = open('saudi_rec_config.yml').read()
cfg = re.sub(r'pretrained_model:.*', f'pretrained_model: ./{pre}', cfg, count=1)
cfg = re.sub(r'eval_batch_step:.*', f'eval_batch_step: [0, {iters}]', cfg, count=1)
open('saudi_rec_config.yml','w').write(cfg)
print(f'{n_train} train crops -> eval + save best every {iters} iters (per epoch)')
!python PaddleOCR/tools/train.py -c saudi_rec_config.yml \
  -o Global.epoch_num=12 Optimizer.lr.warmup_epoch=1 \
     Train.loader.batch_size_per_card=64 Eval.loader.batch_size_per_card=64


## 6. Test on your own images (no export)
On Kaggle you can't pop up a file picker. Two easy ways to get your plate images in:

**A)** Right panel → **Input → + Add Input → Upload** a folder/zip of your CROPPED plate images.
It mounts read-only at `/kaggle/input/<your-dataset-name>/`.

**B)** Or just put images in `/kaggle/working/myplates/` via the file browser.

Set `SRC` below to wherever your images are, then run.


In [ ]:
SRC = '/kaggle/working/myplates'   # <-- change to your /kaggle/input/<name> folder if you added one

from PIL import Image
import numpy as np, os, glob
from plate_spec import split_saudi_plate
os.makedirs('myhalves', exist_ok=True)
imgs = [f for f in glob.glob(f'{SRC}/*') if f.lower().endswith(('.png','.jpg','.jpeg'))]
assert imgs, f'no images found in {SRC} — add them first (see the cell above)'
for p in imgs:
    img = np.array(Image.open(p).convert('RGB'))[:, :, ::-1]
    top, bottom = split_saudi_plate(img)
    stem = os.path.splitext(os.path.basename(p))[0]
    Image.fromarray(bottom[:, :, ::-1]).save(f'myhalves/{stem}__EN.jpg')
    Image.fromarray(top[:, :, ::-1]).save(f'myhalves/{stem}__AR.jpg')
print('halves:', os.listdir('myhalves'))
!python PaddleOCR/tools/infer_rec.py -c saudi_rec_config.yml \
  -o Global.checkpoints=./output/saudi_rec/best_accuracy Global.infer_img=./myhalves


---
## 7. (LATER) Export to ONNX
Only after you're happy with the readings.


In [ ]:
!python PaddleOCR/tools/export_model.py -c saudi_rec_config.yml \
  -o Global.checkpoints=./output/saudi_rec/best_accuracy Global.save_inference_dir=./inference/saudi_rec
import os
mfile = 'inference.json' if os.path.exists('inference/saudi_rec/inference.json') else 'inference.pdmodel'
!paddle2onnx --model_dir ./inference/saudi_rec --model_filename {mfile} \
  --params_filename inference.pdiparams --save_file ./inference/saudi_rec.onnx --opset_version 13
!ls -la inference


---
## 8. Add your REAL Roboflow plates
Your Roboflow project labels **regions** (boxes), not text — so we use it to
*crop* real plates, then the trained model pre-fills a guess you'll correct.

**In Roboflow:** open your project → **Versions → Create New Version**
(no augmentations needed, just click through) → **Download Dataset →
Format: `COCO` → "show download code"**. From that snippet copy your
`api_key` and the `version` number into the cell below, then run it.
Internet is already ON from step 1.

In [ ]:
!pip install -q roboflow
from roboflow import Roboflow
rf = Roboflow(api_key="PASTE_YOUR_API_KEY_HERE")          # <-- from Roboflow's download snippet
project = rf.workspace("tanzim-alam-nueof").project("saudi-plate-tijbb-ul7c7")
version = project.version(1)                               # <-- set to the version you just created
dataset = version.download("coco")
RF_DIR = dataset.location
print("downloaded real dataset to:", RF_DIR)

## 9. Crop the plates + pre-fill guesses
Needs the ONNX model from **step 7** — run step 7 first if you haven't.
This crops every real plate and has the model guess each one.

In [ ]:
!python prepare_real_data.py --coco "{RF_DIR}" --onnx inference/saudi_rec.onnx \
    --dict dataset/saudi_plate_dict.txt --out real_data
from IPython.display import FileLink, display
print("\nClick to download the labeling page:")
display(FileLink("real_data/label_me.html"))

## 10. Correct the guesses, then build real labels
1. Click the **label_me.html** link above to download it (or find it in the
   right-hand **Output** panel).
2. Open it in your browser. Each plate shows the model's guess in an editable
   box, **worst-confidence first**. Fix the wrong ones. Format: `1234 ABC`
   (digits, one space, letters). Leave a box empty to skip a plate you can't read.
3. Click **Download corrections** → you get `corrected.json`.
4. Put `corrected.json` back into Kaggle: right panel → **Input → + Add Input →
   Upload** the file. It mounts under `/kaggle/input/...`. Then run the cell below
   (it finds the file automatically).

In [ ]:
import glob, shutil
hits = glob.glob("/kaggle/working/corrected.json") + glob.glob("/kaggle/input/**/corrected.json", recursive=True)
assert hits, "corrected.json not found — upload it via Input -> Add Input -> Upload, then re-run this cell"
shutil.copy(hits[0], "corrected.json"); print("using", hits[0])
!python build_real_labels.py --crops real_data/crops --corrected corrected.json --out dataset --val-frac 0.2

## 11. Fine-tune on the real plates
Loads your trained model's weights and continues training, mixing the real crops
(oversampled, since there are only a few) with the synthetic data. It then
reports accuracy on a **held-out set of REAL plates** — a far more honest number
than the synthetic accuracy. First cell builds the config; second cell trains.

In [ ]:
import yaml

OVERSAMPLE = 25                                            # repeat each real crop so it counts
real = [l for l in open("dataset/real_train_label.txt") if l.strip()]
with open("dataset/real_train_oversampled.txt","w") as f:
    for _ in range(OVERSAMPLE):
        f.writelines(real)
print(f"{len(real)} real train half-crops x{OVERSAMPLE} = {len(real)*OVERSAMPLE} mixed in")

cfg = yaml.safe_load(open("saudi_rec_config.yml"))
cfg["Global"]["pretrained_model"] = "./output/saudi_rec/best_accuracy"   # load weights, fresh epochs
cfg["Global"]["checkpoints"] = None
cfg["Global"]["save_model_dir"] = "./output/saudi_rec_real"
cfg["Global"]["epoch_num"] = 10
cfg["Optimizer"]["lr"]["learning_rate"] = 0.0001                         # gentle fine-tune LR
cfg["Optimizer"]["lr"]["warmup_epoch"] = 0
cfg["Train"]["dataset"]["label_file_list"] = ["./dataset/train_label.txt","./dataset/real_train_oversampled.txt"]
cfg["Eval"]["dataset"]["label_file_list"] = ["./dataset/real_val_label.txt"]
n_train = sum(1 for l in open("dataset/train_label.txt") if l.strip()) + len(real)*OVERSAMPLE
cfg["Global"]["eval_batch_step"] = [0, max(1, n_train//64)]
yaml.safe_dump(cfg, open("saudi_rec_finetune.yml","w"), allow_unicode=True, sort_keys=False)
print("wrote saudi_rec_finetune.yml — best real-plate model will be ./output/saudi_rec_real/best_accuracy")

In [ ]:
!python PaddleOCR/tools/train.py -c saudi_rec_finetune.yml

## 12. (REAL-ONLY) Prepare the Kaggle character dataset

Add the dataset first: right panel ▸ **+ Add Input** ▸ search **saudi-license-plate-characters** (riotulab) ▸ Add.
This converts all 593 plates into ~978 REAL labeled half-crops (Arabic top + Latin bottom).
**Run step 5 (synthetic training) earlier in this same session first** — the fine-tune below starts from that model.

In [ ]:
import glob, os
# find the dataset root (the folder that holds train/ and test/)
SRC=None
for root,dirs,files in os.walk('/kaggle/input'):
    low=[d.lower() for d in dirs]
    if 'train' in low and 'test' in low and glob.glob(os.path.join(root,'**','*.xml'),recursive=True):
        SRC=root; break
if SRC is None:                                  # fallback: any folder with xml labels
    hits=glob.glob('/kaggle/input/**/*.xml', recursive=True)
    assert hits, "No .xml found — did you Add Input 'saudi-license-plate-characters'?"
    SRC=os.path.dirname(hits[0])
print("dataset root:", SRC)

# make sure the character dictionary exists (normally written during step 3)
if not os.path.exists('dataset/saudi_plate_dict.txt'):
    os.makedirs('dataset', exist_ok=True)
    from plate_spec import dictionary_chars
    open('dataset/saudi_plate_dict.txt','w',encoding='utf-8').write("\n".join(dictionary_chars())+"\n")
    print("wrote dataset/saudi_plate_dict.txt")

!python prepare_char_dataset.py --src "{SRC}" --out dataset --val-frac 0.2
!wc -l dataset/char_train_label.txt dataset/char_val_label.txt

## 13. Fine-tune REAL-ONLY on the Kaggle plates

Starts from the synthetic model (so it already knows every character) and adapts it to real photos.
Best model is saved to `./output/saudi_rec_chars/best_accuracy`. Watch the `acc` printed during eval —
that number is honest real-world accuracy on held-out real plates.

In [ ]:
import yaml
cfg=yaml.safe_load(open("saudi_rec_config.yml"))
cfg["Global"]["pretrained_model"]="./output/saudi_rec/best_accuracy"   # start from synthetic model
cfg["Global"]["checkpoints"]=None
cfg["Global"]["save_model_dir"]="./output/saudi_rec_chars"
cfg["Global"]["epoch_num"]=40
cfg["Optimizer"]["lr"]["learning_rate"]=0.0001                          # gentle fine-tune LR
cfg["Optimizer"]["lr"]["warmup_epoch"]=1
cfg["Train"]["dataset"]["label_file_list"]=["./dataset/char_train_label.txt"]   # REAL ONLY
cfg["Eval"]["dataset"]["label_file_list"]=["./dataset/char_val_label.txt"]
cfg["Train"]["loader"]["batch_size_per_card"]=32
n_train=sum(1 for l in open("dataset/char_train_label.txt") if l.strip())
cfg["Global"]["eval_batch_step"]=[0, max(1, n_train//32)]               # eval ~ every epoch
yaml.safe_dump(cfg, open("saudi_rec_finetune_chars.yml","w"), allow_unicode=True, sort_keys=False)
print(f"{n_train} real train half-crops | wrote saudi_rec_finetune_chars.yml")
print("best model will be at ./output/saudi_rec_chars/best_accuracy")

In [ ]:
!python PaddleOCR/tools/train.py -c saudi_rec_finetune_chars.yml

## 14. DIAGNOSTIC — Latin vs Arabic accuracy

Splits the 76% real-accuracy number into the **Latin line** (the plate ID you actually use) vs the
**Arabic top line**. If Latin is much higher, you're in better shape than the headline number.

In [ ]:
import subprocess
val=[l for l in open("dataset/char_val_label.txt") if l.strip()]
open("dataset/char_val_en.txt","w").writelines([l for l in val if "__EN.jpg" in l])
open("dataset/char_val_ar.txt","w").writelines([l for l in val if "__AR.jpg" in l])
print("EN val crops:", sum("__EN" in l for l in val), "| AR val crops:", sum("__AR" in l for l in val))
for name,f in [("LATIN (English line)","./dataset/char_val_en.txt"),
               ("ARABIC (top line)","./dataset/char_val_ar.txt")]:
    r=subprocess.run(["python","PaddleOCR/tools/eval.py","-c","saudi_rec_finetune_chars.yml",
        "-o","Global.pretrained_model=./output/saudi_rec_chars/best_accuracy",
        f"Eval.dataset.label_file_list=[{f}]"], capture_output=True, text=True)
    acc=[l for l in (r.stdout+r.stderr).splitlines() if "acc:" in l]
    print(f"\n==== {name} ====")
    print(acc[-1] if acc else (r.stdout[-500:]+r.stderr[-500:]))

## 15. (Option B) Fine-tune MIXED — real + synthetic

The real-only model plateaued at 76%. Mixing synthetic back in (real oversampled 15x + synthetic)
usually adds several points by regularizing and covering rare characters. Evaluated on the SAME real
val set, so its accuracy is directly comparable to the 76%. Best model -> ./output/saudi_rec_mix/best_accuracy

In [ ]:
import yaml
OVERSAMPLE=15
real=[l for l in open("dataset/char_train_label.txt") if l.strip()]
open("dataset/char_train_oversampled.txt","w").writelines(real*OVERSAMPLE)
cfg=yaml.safe_load(open("saudi_rec_config.yml"))
cfg["Global"]["pretrained_model"]="./output/saudi_rec/best_accuracy"
cfg["Global"]["checkpoints"]=None
cfg["Global"]["save_model_dir"]="./output/saudi_rec_mix"
cfg["Global"]["epoch_num"]=15
cfg["Optimizer"]["lr"]["learning_rate"]=0.0001
cfg["Optimizer"]["lr"]["warmup_epoch"]=1
cfg["Train"]["dataset"]["label_file_list"]=["./dataset/train_label.txt","./dataset/char_train_oversampled.txt"]
cfg["Eval"]["dataset"]["label_file_list"]=["./dataset/char_val_label.txt"]
n=sum(1 for l in open("dataset/train_label.txt") if l.strip())+len(real)*OVERSAMPLE
cfg["Global"]["eval_batch_step"]=[0,max(1,n//64)]
yaml.safe_dump(cfg,open("saudi_rec_finetune_mix.yml","w"),allow_unicode=True,sort_keys=False)
print(f"synthetic + {len(real)}x{OVERSAMPLE} real = {n} train crops | wrote saudi_rec_finetune_mix.yml")

In [ ]:
!python PaddleOCR/tools/train.py -c saudi_rec_finetune_mix.yml